In [ ]:
import json
import pandas as pd
import numpy as np
import os
import torch

from tabpfn_time_series import TabPFNTSPipeline, TabPFNMode


def root_mean_squared_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
# ============================================================
# CONFIG
# ============================================================
DAYS_JSON = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\dataset_days.json"
DATA_DIR  = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\DataCleaning\clean"
OUT_DIR   = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs"

countries = ["Germany", "Ireland", "Portugal"]
days = ["day1", "day2", "day3", "day4", "day5"]

# present in CSV but NOT used in this univariate setup
features = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation",
]

PRED_LEN = 96
MAX_CONTEXT = 10000

# ============================================================
# LOAD DAY CUTOFFS
# ============================================================
with open(DAYS_JSON, "r") as f:
    dataset_days = json.load(f)

# ============================================================
# INIT TabPFN-TS PIPELINE (LOCAL)
# ============================================================
pipeline = TabPFNTSPipeline(tabpfn_mode=TabPFNMode.LOCAL)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))

rmse_results = []

for country in countries:
    print("Processing country:", country)

    data_path = rf"{DATA_DIR}\dataset_{country.capitalize()}.csv"
    df = pd.read_csv(data_path, index_col="timestamp", parse_dates=True).sort_index()

    households = [c for c in df.columns if c not in features]

    for day in days:
        print("   Day:", day)

        cutoff = pd.to_datetime(dataset_days[country][day])

        predictions_df_all_households = None
        rmse_households = []

        for household in households:
            # ----------------------------
            # 1) Build context_df (univariate only)
            # ----------------------------
            hist_idx = df.index[df.index < cutoff]
            s_train = df.loc[hist_idx, household].astype(float)

            # skip empty / tiny history
            if s_train.dropna().shape[0] < 10:
                continue

            context_df = pd.DataFrame({
                "item_id": household,
                "timestamp": s_train.index,
                "target": s_train.values,
            }).tail(MAX_CONTEXT).reset_index(drop=True)

            # ----------------------------
            # 2) Predict (no future_df => use prediction_length)
            # ----------------------------
            pred_df = pipeline.predict_df(
                context_df=context_df,
                prediction_length=PRED_LEN,
            )

            pred_df_reset = pred_df.reset_index()  # item_id, timestamp back as columns

            # Forecast timestamps for this household
            ts_pred = pd.to_datetime(pred_df_reset["timestamp"])

            # init global predictions df once per day using the forecast timestamps
            if predictions_df_all_households is None:
                predictions_df_all_households = pd.DataFrame(index=ts_pred)

            # Median forecast column is float 0.5
            y_pred = pred_df_reset[0.5].to_numpy()
            predictions_df_all_households[household] = y_pred

            # ----------------------------
            # 3) RMSE aligned by forecast timestamps
            # ----------------------------
            y_true = df.loc[ts_pred, household].to_numpy()

            # if NaNs in truth window, skip rmse for this household
            if np.isnan(y_true).any():
                continue

            rmse_households.append(root_mean_squared_error(y_true, y_pred))

        if predictions_df_all_households is None or len(rmse_households) == 0:
            print(f"      No predictions produced for {country} {day}. Skipping.")
            continue

        avg_rmse_households = float(np.mean(rmse_households))

        rmse_results.append({
            "country": country,
            "day": day,
            "rmse": avg_rmse_households
        })

        output = rf"{OUT_DIR}\TabPFNTS_UNIV_pred_{day}_{country.capitalize()}.csv"
        os.makedirs(os.path.dirname(output), exist_ok=True)
        predictions_df_all_households.to_csv(output, index=True)
        print("      Saved:", output)

# ============================================================
# SUMMARY
# ============================================================
rmse_df = pd.DataFrame(rmse_results)
print("\nPer-day RMSE:")
print(rmse_df)

print("\nCross-validated RMSE per country (mean over days):")
print(rmse_df.groupby("country")["rmse"].mean())

Torch: 2.5.1+cu121
CUDA available: True
Using GPU: NVIDIA RTX 4000 Ada Generation
Processing country: Germany
   Day: day1


GPU 0:: 100%|██████████| 1/1 [00:02<00:00,  2.66s/it]


      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TabPFNTS_UNIV_pred_day1_Germany.csv
   Day: day2


GPU 0:: 100%|██████████| 1/1 [00:03<00:00,  3.54s/it]


      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TabPFNTS_UNIV_pred_day2_Germany.csv
   Day: day3


...

      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TabPFNTS_UNIV_pred_day3_Portugal.csv
   Day: day4


GPU 0:: 100%|██████████| 1/1 [00:03<00:00,  3.53s/it]


      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TabPFNTS_UNIV_pred_day4_Portugal.csv
   Day: day5


GPU 0:: 100%|██████████| 1/1 [00:03<00:00,  3.57s/it]

      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TabPFNTS_UNIV_pred_day5_Portugal.csv

Per-day RMSE:
     country   day         rmse
0    Germany  day1  1205.008324
1    Germany  day2   244.877401
2    Germany  day3   666.962072
3    Germany  day4  1188.971082
4    Germany  day5   256.603703
5    Ireland  day1   782.690022
6    Ireland  day2   484.123467
7    Ireland  day3   538.478691
8    Ireland  day4   735.627778
9    Ireland  day5   424.468266
10  Portugal  day1   387.584484
11  Portugal  day2   228.724707
12  Portugal  day3   288.901765
13  Portugal  day4   352.599470
14  Portugal  day5   196.989750

Cross-validated RMSE per country (mean over days):
country
Germany     712.484516
Ireland     593.077645
Portugal    290.960035
Name: rmse, dtype: float64
